In [5]:
import math

def combinations(n, k):
    """
    Calculates the binomial coefficient "n choose k".
    Returns 0 if k > n or if k or n is negative, which is standard for these formulas.
    """
    if k < 0 or n < 0 or k > n:
        return 0
    return math.comb(n, k)

def get_y_uniform_poly(k, n):
    """
    Computes the inverse Z-polynomial Y(U_k,n)(t) for a uniform matroid.
    
    This function implements the formula from Theorem 1.1 of "The inverse Z-polynomial 
    of a matroid" by Gao, Ruan, and Xie (arXiv:2507.01332v1).
    
    The formula is:
    Y(U_k,n)(t) = sum_{i=0 to floor(k/2)} C(n,i)*C(n-i-1, n-k)*t^i 
                 + sum_{i=0 to floor((k-1)/2)} C(n,i)*C(n-i-1, n-k)*t^(k-i)
                 
    Args:
        k (int): The rank of the uniform matroid.
        n (int): The number of elements in the ground set.
        
    Returns:
        A list of integers representing the coefficients of the polynomial,
        where the index corresponds to the power of t.
    """
    if not isinstance(k, int) or not isinstance(n, int) or k < 1 or n < k:
        raise ValueError("Rank (k) and size (n) must be integers with n >= k >= 1.")

    # The degree of the polynomial is k. Initialize coefficients with zeros.
    coeffs = [0] * (k + 1)

    # --- First Summation ---
    # This sum contributes to the lower-degree terms of the polynomial.
    # for i from 0 to floor(k/2):
    #   coeff_i = C(n, i) * C(n - i - 1, n - k)
    limit1 = k // 2
    for i in range(limit1 + 1):
        term_coeff = combinations(n, i) * combinations(n - i - 1, n - k)
        coeffs[i] += term_coeff

    # --- Second Summation ---
    # This sum contributes to the higher-degree terms of the polynomial.
    # Due to the palindromic nature of Y, this is symmetric to the first sum.
    # for i from 0 to floor((k-1)/2):
    #   power = k - i
    #   coeff_power = C(n, i) * C(n - i - 1, n - k)
    limit2 = (k - 1) // 2
    for i in range(limit2 + 1):
        power = k - i
        term_coeff = combinations(n, i) * combinations(n - i - 1, n - k)
        coeffs[power] += term_coeff
        
    return coeffs

def format_polynomial(coeffs):
    """
    Formats a list of coefficients into a human-readable polynomial string.
    """
    if not coeffs:
        return "0"
        
    poly_str = []
    for i, coeff in enumerate(coeffs):
        if coeff == 0:
            continue
            
        # Handle the term string
        if i == 0:
            term = str(coeff)
        elif i == 1:
            term = f"{coeff}t" if coeff != 1 else "t"
        else:
            term = f"{coeff}t^{i}" if coeff != 1 else f"t^{i}"
            
        poly_str.append(term)
        
    return " + ".join(reversed(poly_str)) if poly_str else "0"

if __name__ == "__main__":
    try:
        print("--- Inverse Z-Polynomial Calculator for Uniform Matroids Y(U_k,n)(t) ---")
        k_str = 1
        n_str = 1
        
        k = int(k_str)
        n = int(n_str)
        
        coefficients = get_y_uniform_poly(k, n)
        polynomial_string = format_polynomial(coefficients)
        
        print("\nResult:")
        print(f"Y(U_{k},{n})(t) = {polynomial_string}")
        
    except ValueError as e:
        print(f"\nError: {e}")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")


--- Inverse Z-Polynomial Calculator for Uniform Matroids Y(U_k,n)(t) ---

Result:
Y(U_1,1)(t) = t + 1


In [26]:
import sys
sys.path.append('..')
from db1 import *

R = PolynomialRing(QQ, 't')
S = PolynomialRing(ZZ, 't')
t = R.gen()

def is_paving(M):
    n = M.size()
    r = M.rank()
    return (len(M.independent_r_sets(r-1)) == binomial(n, r-1))

def q_kl(k, h):
    return kazhdan_lusztig_inverse_uniform(k, h+1) - kazhdan_lusztig_inverse_uniform(k-1, h)

def kl_inverse_fast(M):
    if M.loops(): return R(0)
    k, n = M.rank(), M.size()
    if k == n or k == 0: return R(1)
    if not M.is_connected():
        ans = R(1)
        CC = M.components()
        for N in CC:
            res = M.delete(M.groundset() - N)
            ans = ans * kl_inverse_fast(res)
        return ans

    if is_paving(M):
        return kl_inverse_paving(M)
    if is_paving(M.dual()):
        return kl_inverse_copaving(M)
    """
    if n <= 8 and M.is_connected():
        for i in range(len(mat[n][k])):
            if mat[n][k][i].is_isomorphic(M):
                return ikl[n][k][i]
    """
    LF = M.lattice_of_flats()
    ans = R(0)
    for F in LF:
        if len(F) != n:
            Res = M.delete(M.groundset() - F)
            Con = M.contract(F)
            chi = characteristic_polynomial(Con)(1/t) * t**(Con.rank())
            PPP = kl_inverse_fast(Res)(t) * (-1)**(Res.rank())
            ans = ans + chi * PPP
    assert (t**k * ans(1/t)).numerator() == -ans(t)
    ans = ans.numerator() * (-1)**(k+1)
    return ans.truncate((k+1)//2)

def kazhdan_lusztig_inverse_uniform(k, n):
    if k == n:
        return R(1)
    d = k
    m = n - d
    ans = 0
    for j in range((d-1)//2 + 1):
        ans = ans + m * (d-2*j)/((m+j) * (m+d-j)) * binomial(d, j) * t**j
    return ans * binomial(m+d, d)

def kl_inverse_paving(M):
    assert is_paving(M)
    n = M.size()
    k = M.rank()
    ans = kazhdan_lusztig_inverse_uniform(k, n)
    for H in M.hyperplanes():
        h = len(H)
        if h >= k:
            ans = ans - q_kl(k, h)
    return ans

def kl_inverse_copaving(M):
    assert is_paving(M.dual())
    n = M.size()
    k = M.rank()
    ans = kazhdan_lusztig_inverse_uniform(k, n)
    for H in M.dual().hyperplanes():
        h = len(H)
        if h >= n-k:
            ans = ans - kli_vtilde_dual(n-k, h, n) + kazhdan_lusztig_inverse_uniform(h-n+k+1, h) * kazhdan_lusztig_inverse_uniform(n-h-1, n-h)
    return ans

def kli_vtilde_dual(k, h, n):
    return helper1(n-k, h, n)

def helper1(k, h, n):
    c = n - h
    ans1 = kazhdan_lusztig_inverse_uniform(k, n)
    ans2 = helper2(c, k, n)
    ans3 = kazhdan_lusztig_inverse_uniform(k-c+1, h) * kazhdan_lusztig_inverse_uniform(c-1, c)
    return ans1 - ans2 + ans3

def helper2(c, k, n):
    h = n - c
    ans = 0
    for j in range(k-c+1):
        ans = ans + binomial(n-c, j) * (-1)**(c-1+j) * kazhdan_lusztig_inverse_uniform(c-1, c) * t**(k-c-j+1) * chuly(k-c-j+1, n-c-j)(1/t)
    for i in range(c-1):
        for j in range(k-i):
            ans = ans + binomial(c, i) * binomial(n-c, j) * (-1)**(i+j) * t**(k-i-j) * helper4(c, k, n, i, j)(1/t)
    ans = ans.numerator().truncate((k-1)//2 + 1)
    if ans[0] < 0:
        ans = -ans
    return ans

def helper3(c, k, n):
    ans = 0
    for j in range(k-c+1):
        ans = ans + binomial(n-c, j) * uniformKLpoly(c-1, c) * (-1)**(k-c-j+1) * kazhdan_lusztig_inverse_uniform(k-c-j+1, n-c-j)
    for i in range(c-1):
        for j in range(k-i):
            ans = ans + binomial(c, i) * binomial(n-c, j) * (-1)**(k-i-j) * helper2(c-i, k-i-j, n-i-j)
    return -ans

def helper4(c, k, n, i, j):
    ans = 0
    for l in range(c-i-1):
        ans = ans + (-1)**l * (t-1)**(max(n-i-j-l-1, 0))
    for u in range(n-k-1):
        ans = doit_once(ans)
    return ans

def chuly(a, b):
    ans = (t-1)**b
    for i in range(b-a):
        ans = doit_once(ans)
    return ans

def doit_once(p):
    p = p // t**2
    p = p * t
    p = p - p(1)
    return p

def lorenzo(k, h, n):
    c = n - h
    ans1 = uniformKLpoly(k, n) + uniformKLpoly(k-c+1, h) * uniformKLpoly(c-1, c)
    ans2 = helper3(c, k, n)
    return ans1 - ans2

invkl = kl_inverse_fast

In [27]:
S = PolynomialRing(ZZ, 't')

def conv1(M):
    lattice = M.lattice_of_flats()
    flats = list(lattice)
    crk = lambda F: M.rank() - M.rank(F)
    rk = lambda F: M.rank(F)
    poly = S(0)
    for F in flats:
        poly += t**rk(F) * M.contract(F).lattice_of_flats().kazhdan_lusztig_polynomial()(t)

    return poly

def conv2(M):
    lattice = M.lattice_of_flats()
    flats = list(lattice)
    crk = lambda F: M.rank() - M.rank(F)
    rk = lambda F: M.rank(F)
    poly = S(0)
    for F in flats:
        poly += t**crk(F) * M.delete(M.groundset() - F).lattice_of_flats().kazhdan_lusztig_polynomial()(t)

    return poly

conv1(matroids.Uniform(4,5))

t^4 + 10*t^3 + 20*t^2 + 10*t + 1

In [28]:
conv2(matroids.Uniform(4,5))

t^4 + 5*t^3 + 10*t^2 + 15*t + 1